In [3]:
import numpy as np

class MarketSimulator:
    def __init__(self, beta_matrix, competitor_designs, market_size):
       
        self.betas = beta_matrix
        self.comp_designs = competitor_designs
        self.market_size = market_size
        
        # Pre-calculate competitor utilities since they remain constant
        # Resulting matrix: (num_respondents, num_competitors)
        self.comp_utilities = np.dot(self.betas, self.comp_designs.T)

    def evaluate_product(self, our_design, our_price, our_cost, price_betas): #Setup main function with inputs expected 
        """
        Evaluates a new product configuration generated by NSGA-II.
        
        our_design: A 2D array of shape (1, num_attribute_levels)
        our_price: Float representing the product price
        our_cost: Float representing the product cost
        """
        # Calculate utility for our new product
        # Resulting matrix: (num_respondents, 1)
        feature_utility = np.dot(self.betas, our_design.T)

        #Adding Price negativity
        #price_betas shape: (num_respondents, 1)
        price_utilities = price_betas * our_price
        our_utility = feature_utility + price_utilities
        
        # Combine all utilities into a single matrix
        # Resulting matrix: (num_respondents, num_competitors + 1)
        all_utilities = np.hstack((self.comp_utilities, our_utility))
        
        # Apply Multinomial Logit (MNL) rule
        exp_utilities = np.exp(all_utilities)
        probabilities = exp_utilities / np.sum(exp_utilities, axis=1, keepdims=True)
        
        # Extract the probability column for our product (the last column)
        our_probabilities = probabilities[:, -1]
        
        # Calculate final objectives
        market_share = np.mean(our_probabilities)
        unit_sales = market_share * self.market_size
        profit = unit_sales * (our_price - our_cost)
        
        return market_share, profit

In [7]:
# Dummy survey
np.random.seed(42) 
num_respondents = 100
num_attribute_levels = 7
dummy_betas = np.random.normal(loc=0.0, scale=1.5, size=(num_respondents, num_attribute_levels)) #Generating dummy beta values, to be replaced by input

market_size = int(input("Enter the total market size (e.g., 50000): "))
our_price = float(input("Enter our product price in dollars (e.g., 2.50): "))
our_cost = float(input("Enter our product unit cost in dollars (e.g., 0.85): "))

# Generate negative price sensitivities for 100 respondents
# Average sensitivity is -0.8 utility points per dollar
price_betas = np.random.normal(loc=-0.8, scale=0.2, size=(num_respondents, 1))

# Define competitor prices
# Competitor A ($2.00) and Competitor B ($3.50)
comp_prices = np.array([[2.00], [3.50]]) 

# Competitor A: 1.25mm, 3A, 50 cycles  -> [1, 0, 0,  1, 0,  1, 0]
# Competitor B: 2.50mm, 5A, 100 cycles -> [0, 0, 1,  0, 1,  0, 1]
competitor_matrix = np.array([
    [1, 0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 0, 1]
])

# Initialize the Simulator
simulator = MarketSimulator(dummy_betas, competitor_matrix, market_size)
# Update the competitor utility calculation to include their prices
# (num_respondents, num_competitors) + (num_respondents, 1) * (1, num_competitors)
simulator.comp_utilities += np.dot(price_betas, comp_prices.T)
# Our Product: 2.00mm, 5A, 50 cycles -> [0, 1, 0,  0, 1,  1, 0]
our_new_design = np.array([[0, 1, 0, 0, 1, 1, 0]]) 

# Running the evaluation
market_share, profit = simulator.evaluate_product(our_new_design, our_price, our_cost, price_betas)

# Output
print(f"Product Design String: {our_new_design[0]}")
print(f"Price: ${our_price:.2f} | Cost: ${our_cost:.2f}")
print(f"Calculated Market Share: {market_share * 100:.2f}%")
print(f"Projected Profit: ${profit:,.2f}")

Enter the total market size (e.g., 50000):  500000
Enter our product price in dollars (e.g., 2.50):  3
Enter our product unit cost in dollars (e.g., 0.85):  0.5


Product Design String: [0 1 0 0 1 1 0]
Price: $3.00 | Cost: $0.50
Calculated Market Share: 29.81%
Projected Profit: $372,679.39


In [8]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np

class Optimizer(ElementwiseProblem):
    def __init__(self, sim, price_betas):
        super().__init__(n_var=4, n_obj=2, 
                         xl=np.array([0, 0, 0, 1.0]), 
                         xu=np.array([2, 1, 1, 5.0]))
        self.sim = sim
        self.price_betas = price_betas 

    def _evaluate(self, x, out, *args, **kwargs):
        design = np.zeros((1, 7))
        design[0, int(round(x[0]))] = 1          
        design[0, 3 + int(round(x[1]))] = 1      
        design[0, 5 + int(round(x[2]))] = 1      

        share, profit = self.sim.evaluate_product(design, x[3], 0.85, self.price_betas)
        out["F"] = [-share, -profit]

# --- Execute and Print ---
res = minimize(Optimizer(simulator, price_betas), NSGA2(pop_size=100), ('n_gen', 1000))

print(f"Found {len(res.F)} optimal solutions:\n" + "-"*45)

for share, profit, price in zip(-res.F[:, 0], -res.F[:, 1], res.X[:, 3]):
    print(f"Share: {share*100:5.2f}% | Profit: ${profit:8.2f} | Price: ${price:5.2f}")

Found 100 optimal solutions:
---------------------------------------------
Share: 23.62% | Profit: $350681.98 | Price: $ 3.82
Share: 55.03% | Profit: $41269.53 | Price: $ 1.00
Share: 23.62% | Profit: $350681.98 | Price: $ 3.82
Share: 55.03% | Profit: $41269.53 | Price: $ 1.00
Share: 47.89% | Profit: $165664.49 | Price: $ 1.54
Share: 40.54% | Profit: $250814.83 | Price: $ 2.09
Share: 43.17% | Profit: $224854.65 | Price: $ 1.89
Share: 44.76% | Profit: $206838.19 | Price: $ 1.77
Share: 26.39% | Profit: $347339.85 | Price: $ 3.48
Share: 54.40% | Profit: $53969.78 | Price: $ 1.05
Share: 52.46% | Profit: $91299.13 | Price: $ 1.20
Share: 35.95% | Profit: $284565.51 | Price: $ 2.43
Share: 52.94% | Profit: $82457.16 | Price: $ 1.16
Share: 28.08% | Profit: $342036.79 | Price: $ 3.29
Share: 44.26% | Profit: $212688.97 | Price: $ 1.81
Share: 53.56% | Profit: $70664.50 | Price: $ 1.11
Share: 48.55% | Profit: $156000.99 | Price: $ 1.49
Share: 51.78% | Profit: $103538.87 | Price: $ 1.25
Share: 52.58%

In [9]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np

class Optimizer(ElementwiseProblem):
    def __init__(self, sim, price_betas):
        super().__init__(n_var=4, n_obj=2, 
                         xl=np.array([0, 0, 0, 1.0]), 
                         xu=np.array([2, 1, 1, 5.0]))
        self.sim = sim
        self.price_betas = price_betas 

    def _evaluate(self, x, out, *args, **kwargs):
        design = np.zeros((1, 7))
        design[0, int(round(x[0]))] = 1          
        design[0, 3 + int(round(x[1]))] = 1      
        design[0, 5 + int(round(x[2]))] = 1      

        share, profit = self.sim.evaluate_product(design, x[3], 0.85, self.price_betas)
        out["F"] = [-share, -profit]

# --- Execute and Print ---
res = minimize(Optimizer(simulator, price_betas), NSGA2(pop_size=100), ('n_gen', 1000))

print(f"Found {len(res.F)} optimal solutions:\n" + "-"*45)

for share, profit, price in zip(-res.F[:, 0], -res.F[:, 1], res.X[:, 3]):
    print(f"Share: {share*100:5.2f}% | Profit: ${profit:8.2f} | Price: ${price:5.2f}")

Found 100 optimal solutions:
---------------------------------------------
Share: 23.62% | Profit: $350681.98 | Price: $ 3.82
Share: 55.03% | Profit: $41269.53 | Price: $ 1.00
Share: 23.62% | Profit: $350681.98 | Price: $ 3.82
Share: 55.03% | Profit: $41269.53 | Price: $ 1.00
Share: 54.05% | Profit: $61124.77 | Price: $ 1.08
Share: 53.37% | Profit: $74366.68 | Price: $ 1.13
Share: 24.90% | Profit: $349971.34 | Price: $ 3.66
Share: 44.98% | Profit: $204197.71 | Price: $ 1.76
Share: 27.96% | Profit: $342519.30 | Price: $ 3.30
Share: 34.22% | Profit: $301919.94 | Price: $ 2.61
Share: 52.69% | Profit: $87023.95 | Price: $ 1.18
Share: 32.99% | Profit: $312677.12 | Price: $ 2.75
Share: 31.03% | Profit: $326917.05 | Price: $ 2.96
Share: 45.79% | Profit: $194156.95 | Price: $ 1.70
Share: 41.23% | Profit: $244537.51 | Price: $ 2.04
Share: 52.12% | Profit: $97513.02 | Price: $ 1.22
Share: 28.81% | Profit: $339019.65 | Price: $ 3.20
Share: 54.40% | Profit: $54061.79 | Price: $ 1.05
Share: 39.23% 